In [2]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

development = pd.read_csv("development_300.csv")
golden = pd.read_csv("golden_eval_200.csv")

majority_intent = development["label_intent"].mode()[0]
predictions = [majority_intent] * len(golden)

print("Majority class:", majority_intent)
print("Accuracy:", accuracy_score(golden["label_intent"], predictions))
print(
    classification_report(
        golden["label_intent"],
        predictions,
        zero_division=0
    )
)

Majority class: software_update_performance
Accuracy: 0.355
                               precision    recall  f1-score   support

      account_access_security       0.00      0.00      0.00        15
billing_subscription_purchase       0.00      0.00      0.00        12
                 connectivity       0.00      0.00      0.00        15
             data_backup_loss       0.00      0.00      0.00        15
               feature_how_to       0.00      0.00      0.00        15
            hardware_physical       0.00      0.00      0.00        15
                other_unclear       0.00      0.00      0.00        15
                 out_of_scope       0.00      0.00      0.00        15
            service_complaint       0.00      0.00      0.00        12
  software_update_performance       0.35      1.00      0.52        71

                     accuracy                           0.35       200
                    macro avg       0.04      0.10      0.05       200
               

In [6]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score

development = pd.read_csv("development_300.csv")
golden = pd.read_csv("golden_eval_200.csv")

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_features=20_000,
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(development["opening_text"])
X_test = vectorizer.transform(golden["opening_text"])

model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, development["label_intent"])
predictions = model.predict(X_test)

print("Accuracy:", accuracy_score(golden["label_intent"], predictions))
print(
    "Macro-F1:",
    f1_score(
        golden["label_intent"],
        predictions,
        average="macro"
    )
)
print(
    classification_report(
        golden["label_intent"],
        predictions,
        zero_division=0
    )
)

results = golden[["thread_id", "opening_text", "label_intent"]].copy()
results["predicted_intent"] = predictions
results.to_csv("tfidf_predictions.csv", index=False)

Accuracy: 0.495
Macro-F1: 0.3833212660116405
                               precision    recall  f1-score   support

      account_access_security       0.47      0.53      0.50        15
billing_subscription_purchase       0.00      0.00      0.00        12
                 connectivity       0.67      0.40      0.50        15
             data_backup_loss       0.75      0.40      0.52        15
               feature_how_to       0.33      0.47      0.39        15
            hardware_physical       0.38      0.33      0.36        15
                other_unclear       0.23      0.20      0.21        15
                 out_of_scope       0.71      0.80      0.75        15
            service_complaint       0.00      0.00      0.00        12
  software_update_performance       0.51      0.73      0.60        71

                     accuracy                           0.49       200
                    macro avg       0.41      0.39      0.38       200
                 weighted avg 